In [ ]:
import numpy as np
import dill
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import LineCollection
import matplotlib.patches as patches
import auxiliaries as aux


In [ ]:
class EmptyAgent:
    def __init__(self):
        pass

def load_data(path):
    import dill
    with open(path, 'rb') as f:
        data = dill.load(f)
        
    print(f"Data loaded from {path}.")
    print(f"Number of agents: {data['MAS_parameters']['num_agents']}")
    print(f"MAS type: {data['MAS_parameters']['MAS_type']}")
    
    agents = []
    for agentid in data['agents']:
        agent = EmptyAgent()
        agent.id = agentid
        agent.cl_x = data['agents'][agentid]['cl_x']
        agent.cl_u = data['agents'][agentid]['cl_u']
        agents.append(agent)
        
    return data, agents

def extract_keys(d, parent_key=""):
    keys = set()
    
    if isinstance(d, dict):
        for key, value in d.items():
            full_key = f"{parent_key}.{key}" if parent_key else key
            keys.add(full_key)
            keys.update(extract_keys(value, full_key))
    
    return keys

In [ ]:
colours = [
    "#0072B2", 
    "#D55E00",  
    "#009E73",
    "#CC79A7",  
    "#56B4E9",  
    "#E69F00",  
    "#B22222",  
]

In [ ]:
fname = "harbour_data"
path = f"./data/{fname}.dill"

data, agents = load_data(path)
print("Discretization:", data['MAS_parameters']['h'])
print("Tracking bound:", data['agents'][list(data['agents'].keys())[0]]['tracking_bound'])

In [ ]:
"""Extract and transform data."""
max_sim_time = data['sim_data']['max_sim_time']

if type(data['sim_data']['cooperative_cost']) is list:
    data['sim_data']['cooperative_cost'] = np.vstack(data['sim_data']['cooperative_cost']).flatten()
    data['sim_data']['tracking_cost'] = np.vstack(data['sim_data']['tracking_cost']).flatten()
    data['sim_data']['change_cost'] = np.vstack(data['sim_data']['change_cost']).flatten()
    data['sim_data']['J'] = np.vstack(data['sim_data']['J']).flatten()
    
for agent in agents:
    if type(agent.cl_x) == list:
        agent.cl_x = np.hstack(agent.cl_x)
        agent.cl_u = np.hstack(agent.cl_u)
    

In [ ]:
"""Value function"""
# Plot from t1 to t2.
t1 = 0
t2 = agents[0].cl_x.shape[1]-1

# Select a feasible start time (the end time is controlled below).
t1 = min(t1, max_sim_time+1)

# Draw the evolution in state space:
fig_V, ax_V = plt.subplots(figsize=(10, 6), num='state evolution')

stop_time = data['sim_data']['cooperative_cost'][t1:t2].shape[0]
ax_V.plot(range(t1, min(t2, stop_time)), data['sim_data']['cooperative_cost'][t1:t2], label='cooperative', color=colours[0])
ax_V.plot(range(t1, min(t2, stop_time)), data['sim_data']['tracking_cost'][t1:t2], label='tracking', color=colours[1])
ax_V.plot(range(max(t1,1), min(t2, stop_time)), data['sim_data']['change_cost'][max(t1,1):t2], label='change', color=colours[2])
ax_V.plot(range(t1, min(t2, stop_time)), data['sim_data']['J'][t1:t2], '--', label='J', color=colours[3])
    
ax_V.set_xlabel('time steps')
ax_V.set_title(f'Value function over time')
ax_V.grid(True)
ax_V.legend()

# Set the y-axis to logarithmic scale.
# ax_V.set_yscale('log')

plt.show()


print(f'Value function difference between the first and last time step: {data["sim_data"]["J"][-1] - data["sim_data"]["J"][0]}')
print(f'Value function at start: {data["sim_data"]["J"][0]}')
print(f'Value function at stop:  {data["sim_data"]["J"][-1]}')
print(f'Cooperation cost at stop: {data["sim_data"]["cooperative_cost"][-1]}')
print(f'Tracking cost at stop: {data["sim_data"]["tracking_cost"][-1]}')

In [ ]:
"""Plot the closed-loop state evolution."""
def f(x, y, n, d):
    if n % 2 != 0:
        raise ValueError("n must be even")
    d = np.radians(d)
    x = x - 9.5
    y = y - 3.3
    return ((x*np.cos(d) + y*np.sin(d)) / 3)**n + ((y*np.cos(d) - x*np.sin(d)) / 1.7)**n - 1
x1 = np.linspace(-1, 16, 400)
x2 = np.linspace(1, 9, 400)
X1, X2 = np.meshgrid(x1, x2)
peninsula = f(X1, X2, 8, -40)

# Plot from t1 to t2.
t1 = 0
t2 = np.inf
step = 1

# Select a feasible start and end time.
t1 = min(t1, max_sim_time+1)
t2 = min(t2, agent.cl_x.shape[1] - 1)

# Draw the evolution in state space:
fig_cl, ax_cl = plt.subplots(figsize=(10, 10), num='state evolution')

for i, agent in enumerate(agents):
    ax_cl.plot(agent.cl_x[0, t1 : t2+1:step], agent.cl_x[1,t1 : t2+1:step], label=f'A{agent.id}_x', color=colours[i],
               #marker='o', markersize=2, 
               linewidth=1.0)
    # Mark the initial state with a larger circle.
    ax_cl.plot(agent.cl_x[0,t1], agent.cl_x[1,t1], marker='o', markersize=6, color=colours[i])
    # Mark the final state with a cross.
for i, agent in enumerate(agents):
    ax_cl.plot(agent.cl_x[0,t2], agent.cl_x[1,t2], marker='s', markersize=7, color=colours[i])

xlabel='$x_1$'
ylabel='$x_2$'

ax_cl.grid()

avrt = [[0, 3.5], [4.5, 2.0], [15., 3.5], [15.0, 5.0], [4.5, 8.0], [0.0, 8.0]]
avrt = np.array(avrt)
avrt = np.vstack([avrt, avrt[0]])
ax_cl.plot(avrt[:, 0], avrt[:, 1], 'black', linewidth=1.5)
contour = ax_cl.contour(X1, X2, peninsula, levels=[0], colors='black', linewidths=1.5)

ax_cl.set_xlim(-0.2, 15.2)
ax_cl.set_ylim(1.9, 8.1)
ax_cl.set_aspect('1', adjustable='box')

plt.show()

In [ ]:
"""All states and inputs"""

# Define time range
t1 = 0
t2_state = max_sim_time + 1
t2_input = max_sim_time

tsave = 1600

# Number of states and inputs
num_states = 6
num_inputs = 2

# Agents to plot
agents2plot = agents[:]

# Select feasible start time
t1 = min(t1, max_sim_time + 1)

# Create a figure with multiple subplots (4 rows, 2 columns)
fig, axes = plt.subplots(4, 2, figsize=(12, 12), num='State & Input Evolution')

## --- Plot All 6 States ---
for idx_state in range(num_states):
    ax = axes[idx_state // 2, idx_state % 2]  # Get subplot position
    title_state = f'Closed-loop state $x_{idx_state+1}$ from t = {t1} to t = {t2_state}'

    for i, agent in enumerate(agents):
        if agent not in agents2plot:
            continue
        tf = min(t2_state, agent.cl_x.shape[1] - 1)
        if idx_state == 2:
            ax.plot(range(t1, tf+1), np.degrees(agent.cl_x[idx_state, t1:tf+1]), 
                    color=colours[i], label=f'{agent.id}_x{idx_state+1}', markersize=0, linewidth=2, marker='o')
        else:
            ax.plot(range(t1, tf+1), agent.cl_x[idx_state, t1:tf+1], 
                    color=colours[i], label=f'{agent.id}_x{idx_state+1}', markersize=0, linewidth=2, marker='o')

        # # Write first three states to files.
        # if idx_state == 0 or idx_state == 1 or idx_state == 2:
        #     # Data of first phase.
        #     trajectory_table = ""
        #     for j in range(t1, tsave+1):
        #         if idx_state == 2:
        #             trajectory_table += f"{j} {np.degrees(agent.cl_x[idx_state, j])}\n"
        #         else:
        #             trajectory_table += f"{j} {agent.cl_x[idx_state, j]}\n"

        #     # Write to file (no headers or footers, just the data)
        #     with open(f"./plotdata/vessel_{agent.id}_x{idx_state+1}.tex", "w") as f:
        #         f.write(trajectory_table)

    if np.linalg.norm(ax.get_ylim()) < 1e-8:
        ax.set_ylim(-0.1, 0.1)
        
    ax.grid()
    ax.legend()
    ax.set_title(title_state)
    ax.set_xlabel('time steps')
    ax.set_ylabel(f'$x_{idx_state+1}$')

## --- Plot All 2 Inputs ---
for idx_input in range(num_inputs):
    ax = axes[3, idx_input]  # Get subplot position (last row)
    title_input = f'Closed-loop input $u_{idx_input+1}$ from t = {t1} to t = {t2_input}'

    for i, agent in enumerate(agents):
        if agent not in agents2plot:
            continue
        tf = min(t2_input, agent.cl_u.shape[1] - 1)
        ax.plot(range(t1, tf+1), agent.cl_u[idx_input, t1:tf+1], 
                color=colours[i], label=f'{agent.id}_u{idx_input+1}', markersize=0, linewidth=2, marker='o')

    if np.linalg.norm(ax.get_ylim()) < 1e-8:
        ax.set_ylim(-0.1, 0.1)
        
    ax.grid()
    ax.legend()
    ax.set_title(title_state)
    ax.set_xlabel('time steps')
    ax.set_ylabel(f'$u_{idx_input+1}$')

# Adjust layout and show plot
plt.tight_layout()
plt.show()


In [ ]:
"""Heading"""
idx = 2

# Plot from t1 to t2.
t1 = 0#max_sim_time+1-max(2*T, 5)
t2 = agent.cl_x.shape[1]-1#max_sim_time+1

# Extract the data:
for i, agent in enumerate(agents):
    agent.yT_cl = [data['sim_data']['yT'][f'{agent.id}'][t] for t in range(t1, t2)]

# Draw the evolution in state space:
fig_yT, ax_yT = plt.subplots(figsize=(6, 6), num='cooperation evolution')

for i, agent in enumerate(agents):
    ax_yT.plot(range(t1, tf+1), np.degrees(agent.cl_x[idx, t1:tf+1]), color=colours[i], label=f'{agent.id}_x{idx+1}', markersize=0, linewidth=2, marker='o')
    ax_yT.plot(range(t1, t2), [np.degrees(agent.yT_cl[t][idx]) for t in range(t1, t2)], color=colours[i], markersize=0, linewidth=1.25, marker='x', label=f'{agent.id}_yT{1}', linestyle='--')

if idx == 2:
    ax_yT.set_ylabel(f'heading in degrees')

ax_yT.grid()

plt.show()


In [ ]:
"""Collision"""
# Plot from t1 to t2.
t1 = 0  # max_sim_time+1-max(2*T, 5)
t2 = max_sim_time + 1

fig_d, axes = plt.subplots(1, 2, figsize=(12, 6), num='lateral distance evolution', sharex=True)

ax_d = axes[0]  # Original plot
ax_zoom = axes[1]  # Zoomed-in plot

considered_pairs = []
j = 0  # Counter for colours
for i, agent in enumerate(agents):
    a1 = agent
    tf = min(t2_state, a1.cl_x.shape[1] - 1)
    for a2 in agents:
        if a2 == a1:
            continue
        if (a2.id, a1.id) in considered_pairs or (a1.id, a2.id) in considered_pairs:
            continue
        else:
            considered_pairs.append((a1.id, a2.id))

        distances = []
        for t in range(t1, tf + 1):
            distances.append(np.linalg.norm(a1.cl_x[0:2, t] - a2.cl_x[0:2, t]))

        # Plot on both axes
        for ax in [ax_d, ax_zoom]:
            ax.plot(range(t1, tf + 1), distances, color=colours[j],
                    label=f'$\\Vert z_{{{a1.id[1:]}}} - z_{{{a2.id[1:]}}} \\Vert$',
                    markersize=0, linewidth=2, marker='o')
        j += 1

# Plot boundary line
for ax in [ax_d, ax_zoom]:
    ax.plot(range(t1, tf + 1),
            [data['MAS_parameters']['collision_distance']] * len(range(t1, tf + 1)),
            color='black', label='boundary', linewidth=2, linestyle='--')

# Labels, titles, and grid
ax_d.set_xlabel('$t$')
ax_d.set_ylabel(f'|| p_1 - p_2 ||')
ax_d.set_title(f'Lateral distance from t = {t1} to t = {t2}')
ax_d.grid(True)
ax_d.legend()

# Zoomed-in plot settings
y_min = data['MAS_parameters']['collision_distance'] - 0.05
y_max = data['MAS_parameters']['collision_distance'] + 0.05
ax_zoom.set_ylim(y_min, y_max)
ax_zoom.set_xlabel('$t$')
ax_zoom.set_title('Zoomed-in on collision threshold')
ax_zoom.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
""""2D cooperation output"""
# Plot from t1 to t2.
t1 = 0#max_sim_time+1-max(2*T, 5)
t2 = len(data['sim_data']['yT'][f'{agent.id}'])-1
step = 1

# Extract the data:
for i, agent in enumerate(agents):
    agent.yT_cl = [data['sim_data']['yT'][f'{agent.id}'][t] for t in range(0, t2)]

# Draw the evolution in state space:
fig_yT, ax_yT = plt.subplots(figsize=(6, 6), num='cooperation evolution')

for i, agent in enumerate(agents):
    yT_cl = agent.yT_cl
    ax_yT.plot([yT_cl[t][0] for t in range(t1, t2)], [yT_cl[t][1] for t in range(t1, t2)], color=colours[i], markersize=2, linewidth=0, marker='o', label=f'{agent.id}_yT{2}')
    ax_yT.plot(yT_cl[0][0], yT_cl[0][1], color=colours[i], markersize=6, linewidth=2, marker='o')
    ax_yT.plot(yT_cl[-1][0],yT_cl[-1][1], color=colours[i], markersize=10, linewidth=2, marker='x')

ax_yT.grid()
ax_yT.legend()
ax_yT.set_xlabel('$y_T1$')
ax_yT.set_ylabel('$y_T2$')
ax_yT.set_title('Closed-loop cooperation output on the 2D plane.')

if np.linalg.norm(ax_yT.get_ylim()) < 1e-9:
    ax_yT.set_ylim(-0.1, 0.1)


avrt = [[0, 3.5], [4.5, 2.0], [15., 3.5], [15.0, 5.0], [4.5, 8.0], [0.0, 8.0]]
avrt = np.array(avrt)
avrt = np.vstack([avrt, avrt[0]])
ax_yT.plot(avrt[:, 0], avrt[:, 1], 'black', linewidth=1.5)
ax_yT.contour(X1, X2, peninsula, levels=[0], colors='black', linewidths=1.5)

ax_yT.set_xlim(-0.2, 15.2)
ax_yT.set_ylim(1.9, 8.1)
ax_yT.set_aspect(1, adjustable='box')

plt.show()